In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
         (os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install -q transformers librosa

In [3]:
import os
import random
import shutil
import numpy as np
import pandas as pd
import torch
import librosa
from tqdm import tqdm
from transformers import ASTFeatureExtractor, ASTForAudioClassification

print("All imports done!")

2026-03-27 16:16:48.800544: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774628209.006351      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774628209.063867      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774628209.554922      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774628209.554969      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774628209.554972      24 computation_placer.cc:177] computation placer alr

All imports done!


In [4]:
SAMPLE_RATE = 16000
DURATION    = 20
MAX_LENGTH  = SAMPLE_RATE * DURATION
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"

GENRES   = ["blues","classical","country","disco","hiphop",
            "jazz","metal","pop","reggae","rock"]
label2id = {g: i for i, g in enumerate(GENRES)}
id2label = {i: g for g, i in label2id.items()}
NUM_LABELS = len(GENRES)

print("Device    :", DEVICE)
print("Num Labels:", NUM_LABELS)

Device    : cuda
Num Labels: 10


In [5]:

import os
import shutil

src = "/kaggle/input/datasets/anshusharma52/gen-ai-project-anshu/best_model_phase2 (4).pth"
MODEL_PATH = "/kaggle/working/best_model_phase2.pth"

if not os.path.exists(src):
    print("Source file not found:", src)
else:
    shutil.copy(src, MODEL_PATH)
    print("Model copied successfully!")

print("Model path exists:", os.path.exists(MODEL_PATH))

Model copied successfully!
Model path exists: True


In [6]:
BASE_PATH = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"


src        = "/kaggle/input/datasets/anshusharma52/my-gen-ai-project1/best_model_phase2 (4).pth"
MODEL_PATH = "/kaggle/working/best_model_phase2.pth"

if not os.path.exists(MODEL_PATH):
    shutil.copy(src, MODEL_PATH)
    print("Model copied successfully!")

print("Base path exists :", os.path.exists(BASE_PATH))
print("Model path exists:", os.path.exists(MODEL_PATH))

Base path exists : True
Model path exists: True


In [7]:
def load_audio(path, sr=SAMPLE_RATE):
    audio, _ = librosa.load(path, sr=sr, mono=True)
    return audio.astype(np.float32)

def crop_or_pad(audio, max_len=MAX_LENGTH):
    if len(audio) >= max_len:
        start = random.randint(0, len(audio) - max_len)
        return audio[start : start + max_len]
    return np.pad(audio, (0, max_len - len(audio)))

def normalize(audio):
    return audio / (np.max(np.abs(audio)) + 1e-6)

print("Audio helpers ready!")

Audio helpers ready!


In [8]:
feature_extractor = ASTFeatureExtractor.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593"
)
print("Feature extractor loaded!")

preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Feature extractor loaded!


In [9]:
model = ASTForAudioClassification.from_pretrained(
    "MIT/ast-finetuned-audioset-10-10-0.4593",
    num_labels             = NUM_LABELS,
    id2label               = id2label,
    label2id               = label2id,
    ignore_mismatched_sizes = True
)

model.load_state_dict(
    torch.load(MODEL_PATH, map_location=DEVICE)
)
model.to(DEVICE)
model.eval()

print("Model loaded successfully on", DEVICE)
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded successfully on cuda
Total params: 86,196,490


In [10]:
def predict_with_tta(model, audio, feature_extractor, n_tta=5):
    all_probs = []

    for _ in range(n_tta):
        cropped = crop_or_pad(audio)
        cropped = normalize(cropped)
        inputs  = feature_extractor(
            cropped,
            sampling_rate  = SAMPLE_RATE,
            return_tensors = "pt"
        )
        input_values = inputs["input_values"].to(DEVICE)

        with torch.no_grad():
            outputs = model(input_values=input_values)
            probs   = torch.softmax(outputs.logits, dim=1)
            all_probs.append(probs.cpu().numpy())

    avg_probs = np.mean(all_probs, axis=0)
    return np.argmax(avg_probs, axis=1)[0]

print("TTA function ready!")

TTA function ready!


In [11]:
test_df = pd.read_csv(os.path.join(BASE_PATH, "test.csv"))

print(f"Test samples : {len(test_df)}")
print(f"Columns      : {test_df.columns.tolist()}")
print(test_df.head())

Test samples : 3020
Columns      : ['id', 'filename']
   id              filename
0   1  mashups/song0001.wav
1   2  mashups/song0002.wav
2   3  mashups/song0003.wav
3   4  mashups/song0004.wav
4   5  mashups/song0005.wav


In [12]:
all_ids   = []
all_preds = []

for idx in tqdm(range(len(test_df)), desc="Inference with TTA"):
    row   = test_df.iloc[idx]
    path  = os.path.join(BASE_PATH, row["filename"])
    audio = load_audio(path)
    pred  = predict_with_tta(model, audio, feature_extractor, n_tta=5)
    all_ids.append(row["id"])
    all_preds.append(id2label[pred])

print(f"\nInference done! Total: {len(all_preds)} predictions")

Inference with TTA: 100%|██████████| 3020/3020 [27:41<00:00,  1.82it/s]


Inference done! Total: 3020 predictions


In [13]:
submission = pd.DataFrame({
    "id"   : all_ids,
    "genre": all_preds
})

submission.to_csv("/kaggle/working/submission.csv", index=False)

print("Submission saved!")
print(f"Total predictions  : {len(submission)}")
print("\nGenre distribution :")
print(submission["genre"].value_counts())
print("\nSample predictions :")
print(submission.head(10))


Submission saved!
Total predictions  : 3020

Genre distribution :
genre
rock         404
pop          392
hiphop       344
jazz         312
metal        300
blues        278
reggae       270
classical    250
disco        248
country      222
Name: count, dtype: int64

Sample predictions :
   id      genre
0   1        pop
1   2  classical
2   3      disco
3   4      metal
4   5    country
5   6        pop
6   7       rock
7   8        pop
8   9        pop
9  10      disco
